<center><h1>Yang_Yuxin_HW3</h1></center>

Name: Yuxin Yang  
USC ID: 1505451394

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

In [1]:
import io
import re
import zipfile
from urllib.request import urlopen

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

Get the AReM Data Set

In [5]:
# Download the original recording files directly from UCI.
url = "https://archive.ics.uci.edu/static/public/366/activity+recognition+system+based+on+multisensor+data+fusion+arem.zip"
with urlopen(url, timeout=60) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))

series_names = ["avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
recordings = {}
cleaned_files = []
missing_samples = {}

files = [name for name in archive.namelist()
         if re.fullmatch(r"[^/]+/dataset\d+\.csv", name)]
files.sort(key=lambda name: (name.split("/")[0],
                            int(re.search(r"dataset(\d+)", name).group(1))))

for name in files:
    text = archive.read(name).decode("utf-8-sig")
    lines = [line.strip() for line in text.splitlines()
             if line.strip() and not line.startswith("#")]
    # A few files use whitespace or have a trailing comma: normalize both.
    if any("," not in line or line.endswith(",") for line in lines):
        cleaned_files.append(name)
    rows = [re.split(r"[,\s]+", line.rstrip(",")) for line in lines]
    data = pd.DataFrame(rows, columns=["time"] + series_names).astype(float)
    expected_times = np.arange(480) * 250
    missing = sorted(set(expected_times) - set(data["time"]))
    if missing:
        missing_samples[name] = missing
    assert data.shape[1] == 7 and not data["time"].duplicated().any()
    assert missing == ([13500] if name == "sitting/dataset8.csv" else []), (name, missing)
    assert len(data) == 480 - len(missing), (name, data.shape)
    assert np.isfinite(data.to_numpy()).all(), name
    assert data["time"].is_monotonic_increasing, name
    recordings[name] = data[series_names]

# Check the folders, files, and dimensions.
checks = []

for filename, data in recordings.items():
    checks.append({
        "activity": filename.split("/")[0],
        "file": filename,
        "time_series": data.shape[1],
        "time_points": data.shape[0]
    })

checks = pd.DataFrame(checks)

summary = checks.groupby("activity").agg(
    instances=("file", "count"),
    time_series_per_instance=("time_series", "min"),
    minimum_time_points=("time_points", "min"),
    maximum_time_points=("time_points", "max")
)

display(summary)

print("Number of activity folders:", checks["activity"].nunique())
print("Total instances:", len(checks))
print("All instances have 6 time series:",
      checks["time_series"].eq(6).all())
print("Time-series names:", list(recordings.values())[0].columns.tolist())

# Show any files that do not contain exactly 480 observations.
print("\nFiles with a different number of observations:")
display(checks.loc[checks["time_points"] != 480])

,instances,time_series_per_instance,minimum_time_points,maximum_time_points
activity,,,,
bending1,7,6,480,480
bending2,6,6,480,480
cycling,15,6,480,480
lying,15,6,480,480
sitting,15,6,479,480
standing,15,6,480,480
walking,15,6,480,480


Number of activity folders: 7
Total instances: 88
All instances have 6 time series: True
Time-series names: ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']

Files with a different number of observations:


,activity,file,time_series,time_points
50,sitting,sitting/dataset8.csv,6,479


### (b) Test and Train Data

In [6]:
metadata = []
for instance, name in enumerate(files, start=1):
    activity = name.split("/")[0]
    dataset_id = int(re.search(r"dataset(\d+)", name).group(1))
    test_limit = 2 if activity in ["bending1", "bending2"] else 3
    split = "test" if dataset_id <= test_limit else "train"
    metadata.append([instance, activity, dataset_id, split, name])

metadata = pd.DataFrame(metadata, columns=["Instance", "activity", "dataset_id", "split", "file"])
train_files = metadata.loc[metadata["split"] == "train", "file"].tolist()
test_files = metadata.loc[metadata["split"] == "test", "file"].tolist()
assert len(train_files) == 69 and len(test_files) == 19
assert set(train_files).isdisjoint(test_files)
print("Training recordings:", len(train_files))
print("Test recordings:", len(test_files))
display(pd.crosstab(metadata["activity"], metadata["split"]))

Training recordings: 69
Test recordings: 19


split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12


### (c) Feature Extraction

#### i. Research

Common time-domain features include:

- **Location:** mean, median, minimum, maximum, and quartiles.
- **Spread:** standard deviation, variance, range, and interquartile range.
- **Shape:** skewness and kurtosis.
- **Magnitude:** root mean square and sum of squared values (energy).
- **Changes over time:** mean absolute change, number of peaks, threshold crossings, and autocorrelation at selected lags.

Source: [tsfresh feature documentation](https://tsfresh.readthedocs.io/en/stable/text/list_of_features.html).

#### ii. Extraction

The archive has one missing observation in `sitting/dataset8.csv` (13,500 ms). I use its 479 available rows without inventing a value; all other recordings have 480 rows. Whitespace delimiters and trailing commas are cleaned when reading. Features are kept in their original units, and standard deviations use `ddof=1`. Series 1 through 6 follow the order in `series_names`.

In [7]:
feature_rows = []
for name in files:
    data = recordings[name]
    row = {}
    for j, column in enumerate(series_names, start=1):
        x = data[column]
        row[f"min{j}"] = x.min()
        row[f"max{j}"] = x.max()
        row[f"mean{j}"] = x.mean()
        row[f"median{j}"] = x.median()
        row[f"std{j}"] = x.std(ddof=1)
        row[f"1st_quart{j}"] = x.quantile(0.25)
        row[f"3rd_quart{j}"] = x.quantile(0.75)
    feature_rows.append(row)

features = pd.DataFrame(feature_rows, index=metadata["Instance"])
features.index.name = "Instance"
assert features.shape == (88, 42)
assert np.isfinite(features.to_numpy()).all()

# Keep labels and split separately from the 42 numeric features.
train_mask = metadata["split"].eq("train").to_numpy()
X_train = features.loc[train_mask].copy()
X_test = features.loc[~train_mask].copy()
y_train = metadata.loc[train_mask, "activity"].to_numpy()
y_test = metadata.loc[~train_mask, "activity"].to_numpy()
print("Feature table:", features.shape)
print("Training features:", X_train.shape, "Test features:", X_test.shape)
display(features.round(3))

Feature table: (88, 42)
Training features: (69, 42) Test features: (19, 42)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,median2,std2,1st_quart2,3rd_quart2,min3,max3,mean3,median3,std3,1st_quart3,3rd_quart3,min4,max4,mean4,median4,std4,1st_quart4,3rd_quart4,min5,max5,mean5,median5,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,37.25,45.00,40.625,40.500,1.477,39.250,42.000,0.0,1.30,0.359,0.430,0.323,0.000,0.500,4.00,29.50,19.041,19.250,4.463,16.000,23.250,0.0,7.23,0.833,0.500,0.966,0.000,1.120,27.25,38.25,34.311,35.000,2.188,33.000,36.000,0.00,1.92,0.571,0.430,0.583,0.000,1.300
2,38.00,45.67,42.813,42.500,1.436,42.000,43.670,0.0,1.22,0.372,0.470,0.289,0.000,0.500,2.00,29.50,20.097,21.000,3.894,19.000,22.250,0.0,5.76,0.876,0.500,1.047,0.000,1.145,27.67,38.50,33.025,33.000,1.995,32.000,34.500,0.00,3.11,0.571,0.430,0.601,0.000,1.300
3,35.00,47.40,43.954,44.330,1.559,43.000,45.000,0.0,1.70,0.426,0.470,0.339,0.000,0.500,6.50,29.75,22.122,23.000,3.031,19.750,24.000,0.0,4.44,0.497,0.430,0.551,0.000,0.830,29.00,38.50,35.588,36.000,2.000,35.362,36.500,0.00,1.79,0.493,0.430,0.514,0.000,0.940
4,33.00,47.75,42.180,43.500,3.671,39.150,45.000,0.0,3.00,0.696,0.500,0.631,0.000,1.120,8.50,30.00,22.184,23.000,3.810,20.500,24.372,0.0,5.15,0.990,0.830,0.954,0.430,1.300,20.00,38.67,33.494,35.000,3.849,30.458,36.330,0.00,2.18,0.614,0.500,0.524,0.000,1.000
5,33.00,45.75,41.678,41.750,2.243,41.330,42.750,0.0,2.83,0.536,0.500,0.405,0.430,0.710,3.00,28.25,19.007,19.125,4.087,16.500,22.062,0.0,6.42,0.842,0.500,0.929,0.430,1.120,23.67,37.50,29.857,30.000,2.411,28.458,31.250,0.00,1.79,0.383,0.430,0.389,0.000,0.500
6,37.00,48.00,43.455,43.250,1.386,42.500,45.000,0.0,1.58,0.378,0.470,0.316,0.000,0.500,5.75,27.00,15.793,15.000,3.848,13.000,18.270,0.0,10.03,0.849,0.500,0.996,0.430,1.120,8.00,33.50,23.035,23.500,2.489,22.250,24.000,0.00,5.26,0.680,0.500,0.623,0.430,0.870
7,36.25,48.00,43.969,44.500,1.618,43.310,44.670,0.0,1.50,0.413,0.470,0.263,0.430,0.500,1.50,26.33,15.868,16.250,3.742,14.250,18.000,0.0,5.17,0.666,0.470,0.789,0.000,0.940,11.33,30.75,22.104,21.670,3.318,20.500,23.750,0.00,2.96,0.555,0.490,0.488,0.000,0.830
8,12.75,51.00,24.563,24.250,3.738,23.188,26.500,0.0,6.87,0.591,0.430,0.837,0.000,0.710,0.00,25.33,19.121,20.250,4.300,18.000,22.000,0.0,6.76,0.737,0.470,0.927,0.000,0.870,10.67,30.00,23.493,23.750,3.694,20.500,27.000,0.00,4.97,0.700,0.500,0.694,0.430,0.870
9,0.00,42.75,27.465,28.000,3.584,25.500,30.000,0.0,7.76,0.450,0.430,0.767,0.000,0.500,7.50,35.00,20.843,20.750,3.826,18.000,22.500,0.0,5.76,0.780,0.500,0.904,0.000,0.940,0.00,33.00,17.618,18.000,5.054,15.000,20.750,0.00,6.76,1.122,0.830,1.012,0.470,1.300


#### iii. Standard Deviation

The standard deviation below measures variation **across the 88 recordings** for each extracted feature. This differs from `std1`, ..., `std6`, which measure variation within a recording. Both use sample standard deviation (`ddof=1`).

I use 10,000 bootstrap samples of 88 recordings, drawn with replacement, and take the 5th and 95th percentiles of the resulting standard deviations. Entire feature rows are resampled together. This is a 90% percentile bootstrap interval, assuming recordings are independent. All 88 recordings are used as requested for this descriptive analysis; later model fitting should use training data only.

In [8]:
rng = np.random.default_rng(552)
B = 10000
values = features.to_numpy()
bootstrap_std = np.empty((B, values.shape[1]))

for b in range(B):
    indices = rng.integers(0, len(values), size=len(values))
    bootstrap_std[b] = values[indices].std(axis=0, ddof=1)

ci = np.quantile(bootstrap_std, [0.05, 0.95], axis=0)
std_summary = pd.DataFrame({
    "standard_deviation": features.std(ddof=1),
    "90% CI lower": ci[0],
    "90% CI upper": ci[1]
})
assert std_summary.shape == (42, 3)
assert (std_summary["90% CI lower"] <= std_summary["90% CI upper"]).all()
display(std_summary.round(4))

,standard_deviation,90% CI lower,90% CI upper
min1,9.5700,8.2622,10.7614
max1,4.3944,3.3280,5.2849
mean1,5.3357,4.7040,5.8864
median1,5.4401,4.7903,6.0043
std1,1.7722,1.5722,1.9500
1st_quart1,6.1536,5.5816,6.6524
3rd_quart1,5.1389,4.3319,5.8409
min2,0.0000,0.0000,0.0000
max2,5.0627,4.6228,5.4022
mean2,1.5742,1.3941,1.7068


#### iv. Select Features

I select **mean, standard deviation, and third quartile**. Mean describes the overall signal level, while standard deviation measures variability. The third quartile describes the upper part of the signal distribution and is less sensitive to extreme outliers than the maximum. These features may help distinguish activities with different signal levels and variability.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic model has training RSS **less than or equal to** that of the linear model. The linear model is a special case of the cubic model with $\beta_2=\beta_3=0$. Least squares minimizes RSS over a larger set of possible fits, so the extra terms cannot increase training RSS, even when the true relationship is linear.

### (b) Linear Test

We generally expect the **linear model to have lower test RSS** when the true relationship is linear. The cubic model adds unnecessary flexibility and usually has higher estimation variance, so it may fit training noise. This is an expectation over samples, not a guarantee for every particular test set.

### (c) Not Linear Train

The cubic model again has training RSS **less than or equal to** the linear model's RSS. The nesting argument in (a) still applies, regardless of the true relationship.

### (d) Not Linear Testing

There is **not enough information to determine which model has lower expected test RSS**. The cubic model can reduce bias if the true relationship has curvature that it approximates well, but it increases variance. With mild nonlinearity the linear model may do better; with stronger suitable nonlinearity the cubic model may do better.
